In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import csv
import time
from urllib.parse import urljoin
import re

In [ ]:
def scrape_soidb(csv_filename):
    urls = [
        "https://www.soidb.com/bangkok/fire/list.html",
        "https://www.soidb.com/bangkok/police/list.html",
        "https://www.soidb.com/bangkok/government/list.html"
    ]
    headers = ["name", "road", "district", "province"]
    
    request_headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }

    with open(csv_filename, mode='w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(headers)

        for start_url in urls:
            print(f"--- Starting Category: {start_url} ---")
            current_url = start_url
            
            while True:
                print(f"Scraping: {current_url}")
                try:
                    response = requests.get(current_url, headers=request_headers, timeout=10)
                    if response.status_code != 200:
                        print(f"Failed to load page: {response.status_code}")
                        break

                    soup = BeautifulSoup(response.content, "html.parser")

                    items = soup.find_all("div", class_="list_main")

                    if not items:
                        print("No items found on this page.")

                    for item in items:

                        name_tag = item.find("div", class_="sd_link", itemprop="name")
                        name = name_tag.get_text(strip=True) if name_tag else ""

                        road = ""
                        district = ""
                        province = ""

                        address_span = item.find("span", itemprop="address")
                        if address_span:
                            road_tag = address_span.find("span", itemprop="streetAddress")
                            if road_tag:
                                road = road_tag.get_text(strip=True)
                            
                            district_tag = address_span.find("span", itemprop="addressLocality")
                            if district_tag:
                                district = district_tag.get_text(strip=True)
                            
                            province_tag = address_span.find("span", itemprop="addressRegion")
                            if province_tag:
                                province = province_tag.get_text(strip=True)

                        writer.writerow([name, road, district, province])

                    next_link = soup.find("a", string=re.compile(r"Next", re.IGNORECASE))
                    
                    if next_link and 'href' in next_link.attrs:

                        next_url = next_link['href']
                        current_url = urljoin(start_url, next_url)
                        

                        time.sleep(1)
                    else:
                        print("No 'Next' page found. Moving to next category.")
                        break

                except Exception as e:
                    print(f"Error occurred: {e}")
                    break
    
    print(f"\nDone! Data saved to {csv_filename}")

In [ ]:
scrape_soidb("department_data.csv")